# S&P 500 Intelligence Oracle: Phase 4
## Machine Learning & Evaluation

### Objectives
1. **Feature Engineering**: Build a feature set from technical indicators, FNSPID sentiment, and an autoregressive lag — avoiding look-ahead bias (no same-day `High`/`Low`).
2. **Train/Test Split**: Chronological split (no shuffle) to respect time-series ordering.
3. **Baseline**: Establish a naive "always predict zero return" benchmark to contextualise model performance.
4. **Random Forest Regressor**: Predict the daily percentage return of the S&P 500 close price.
5. **Evaluation**: MAE, RMSE, R², directional accuracy, feature importance, and prediction plots.

In [6]:
from pyspark.sql import SparkSession
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

spark = SparkSession.builder.appName("SP500_Modeling").getOrCreate()

df = spark.read.parquet("../data/processed/integrated_data_parquet").toPandas()

# Ensure chronological order
df = df.sort_values("Date").reset_index(drop=True)

print(f"Loaded {len(df):,} rows spanning {df['Date'].min()} → {df['Date'].max()}")
print(f"Columns: {df.columns.tolist()}")

Loaded 24,532 rows spanning 1927-12-30 → 2025-08-29
Columns: ['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'Close_7D_Avg', 'Close_30D_Avg', 'Daily_Sentiment_Score', 'Daily_Article_Count']


### Step 1 — Feature Engineering

Raw moving average prices (`Close_7D_Avg`, `Close_30D_Avg`) are non-stationary — a value of 1,000 means something completely different in 1960 vs 2020, so a model trained on historical prices fails on modern prices. We replace them with **price-relative ratios** that are scale-invariant across the full 97-year range.

| Feature | Formula | Description |
|---|---|---|
| `Price_vs_7D_MA` | `Close / Close_7D_Avg − 1` | Short-term momentum: how far price sits above/below its 7D average |
| `Price_vs_30D_MA` | `Close / Close_30D_Avg − 1` | Medium-term momentum: same for 30D average |
| `Volatility_20D` | Rolling 20D std of daily returns | Market regime signal — scale-invariant by definition |
| `Daily_Sentiment_Score` | VADER mean across all articles that day | FNSPID news sentiment |
| `Daily_Article_Count` | Count of articles published that day | Market attention proxy |
| `Prev_Return` | Yesterday's percentage return | Autoregressive signal |

In [7]:
# Target: daily percentage return
df["Target_Return"] = df["Close"].pct_change()

# Price-relative momentum ratios (scale-invariant across 97 years of prices)
df["Price_vs_7D_MA"]  = df["Close"] / df["Close_7D_Avg"]  - 1
df["Price_vs_30D_MA"] = df["Close"] / df["Close_30D_Avg"] - 1

# Rolling 20-day return volatility (regime signal, already scale-invariant)
df["Volatility_20D"] = df["Target_Return"].rolling(20).std()

# Autoregressive feature: previous day's return
df["Prev_Return"] = df["Target_Return"].shift(1)

FEATURES = [
    "Price_vs_7D_MA",
    "Price_vs_30D_MA",
    "Volatility_20D",
    "Daily_Sentiment_Score",
    "Daily_Article_Count",
    "Prev_Return",
]

df = df.dropna(subset=FEATURES + ["Target_Return"])

print(f"Rows after dropping NaNs: {len(df):,}")
print(f"\nFeature statistics (all should be small, scale-invariant numbers):")
df[FEATURES].describe().round(4)

,Price_vs_7D_MA,Price_vs_30D_MA,Volatility_20D,Daily_Sentiment_Score,Daily_Article_Count,Prev_Return
count,24512.0000,24512.0000,24512.0000,24512.0000,24512.0000,24512.0000
mean,0.0008,0.0038,0.0098,0.0078,19.9926,0.0003
std,0.0159,0.0364,0.0069,0.0286,61.8637,0.0119
min,-0.2307,-0.3260,0.0015,-0.2174,0.0000,-0.2047
25%,-0.0064,-0.0128,0.0058,0.0000,0.0000,-0.0045
50%,0.0020,0.0076,0.0078,0.0000,0.0000,0.0005
75%,0.0091,0.0239,0.0112,0.0000,0.0000,0.0055
max,0.1582,0.4249,0.0670,0.7994,975.0000,0.1661


### Step 2 — Chronological Train / Test Split

In [8]:
split_idx = int(len(df) * 0.8)

X_train = df[FEATURES].iloc[:split_idx]
X_test  = df[FEATURES].iloc[split_idx:]
y_train = df["Target_Return"].iloc[:split_idx]
y_test  = df["Target_Return"].iloc[split_idx:]
dates_test = df["Date"].iloc[split_idx:]

print(f"Train: {len(X_train):,} rows  ({df['Date'].iloc[0]} → {df['Date'].iloc[split_idx-1]})")
print(f"Test:  {len(X_test):,} rows  ({df['Date'].iloc[split_idx]} → {df['Date'].iloc[-1]})")

Train: 19,609 rows  (1928-01-30 → 2006-03-06)
Test:  4,903 rows  (2006-03-07 → 2025-08-29)


### Step 3 — Baseline & Random Forest Model

In [9]:
def evaluate(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    dir_acc = np.mean(np.sign(y_true) == np.sign(y_pred))
    print(f"{'─'*40}")
    print(f" {name}")
    print(f"{'─'*40}")
    print(f"  MAE:                  {mae:.6f}")
    print(f"  RMSE:                 {rmse:.6f}")
    print(f"  R²:                   {r2:.4f}")
    print(f"  Directional accuracy: {dir_acc:.2%}")
    return {"MAE": mae, "RMSE": rmse, "R2": r2, "DirAcc": dir_acc, "preds": y_pred}

# Naive baseline: always predict zero return
baseline_preds = np.zeros(len(y_test))
baseline = evaluate("Baseline (predict 0)", y_test, baseline_preds)

# Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)
rf_results = evaluate("Random Forest Regressor", y_test, rf_preds)

────────────────────────────────────────
 Random Forest Regressor
────────────────────────────────────────
  MAE:                  0.005368
  RMSE:                 0.008120
  R²:                   0.5701
  Directional accuracy: 76.22%


### Step 4 — Visualisations

In [10]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("S&P 500 Intelligence Oracle — Model Evaluation", fontsize=14, fontweight="bold")

# 1. Feature importance
importances = pd.Series(rf.feature_importances_, index=FEATURES).sort_values()
axes[0].barh(importances.index, importances.values, color="steelblue")
axes[0].set_title("Feature Importance")
axes[0].set_xlabel("Mean decrease in impurity")

# 2. Actual vs predicted returns (test period, last 252 trading days for readability)
plot_n = min(252, len(y_test))
axes[1].plot(dates_test.values[-plot_n:], y_test.values[-plot_n:],
             label="Actual", alpha=0.7, linewidth=0.8)
axes[1].plot(dates_test.values[-plot_n:], rf_preds[-plot_n:],
             label="Predicted", alpha=0.7, linewidth=0.8)
axes[1].set_title("Actual vs Predicted Returns (last year of test set)")
axes[1].set_xlabel("Date")
axes[1].set_ylabel("Daily Return")
axes[1].legend()
axes[1].tick_params(axis="x", rotation=45)

# 3. Prediction error distribution
errors = y_test.values - rf_preds
axes[2].hist(errors, bins=60, color="coral", edgecolor="white")
axes[2].axvline(0, color="black", linestyle="--", linewidth=1)
axes[2].set_title("Prediction Error Distribution")
axes[2].set_xlabel("Actual − Predicted")
axes[2].set_ylabel("Frequency")

plt.tight_layout()
plt.savefig("../reports/model_evaluation.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved to reports/model_evaluation.png")

Plot saved to reports/model_evaluation.png
